# ⏱️ Simulador Visual: La Máquina del Tiempo
**Objetivo:** Viajar a un punto específico del pasado, entrenar las leyes físicas de SINDy usando solo la información disponible hasta ese instante, y luego **superponer la Proyección Matemática** contra lo que *realmente* terminó sucediendo en el mercado.
Esto permite inspeccionar de forma empírica y visual la fidelidad o divergencia del modelo.

In [ ]:
import sys
import os
import warnings
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"

from src.ui.market_loader import MarketLoader
from src.quant_engine.sensor import SpectralAnalyzer
from src.quant_engine.blender import ContinuousBlender
from src.quant_engine.nervous import RegimeShiftDetector
from src.quant_engine.physics import PhysicsDiscoverer
from src.quant_engine.auto_tuner import CUSUMAutoTuner
from src.quant_engine.auto_tuner_predictive import PredictiveAutoTuner

### 1. Descarga del Espectro Completo de Datos

In [ ]:
ticker = "GC=F"
period = "1y"
interval = "1h"

print(f"Descargando datos históricos completos de {ticker}...")
df_mercado = MarketLoader.load_ticker_data(ticker, period=period, interval=interval)
print(f"Total Velas Disponibles en la Base: {len(df_mercado)}")

### 2. Configurar la "Fecha del Corte" (Día Cero)
Aquí decides a qué momento del pasado quieres viajar. Puedes establecerlo por porcentaje (Ej: `0.5` = justo a la mitad del historial).

In [ ]:
# --- CONFIGURACIÓN DE LA MÁQUINA DEL TIEMPO ---
# Proporción del historial a usar como "pasado conocido"
# 0.8 = usar el 80% inicial de los datos para entrenar, predecir el 20% restante a ciegas.
PORCENTAJE_PASADO = 0.95
#Ahora me surge una duda, porque no sé asi a ala hora de hacer la validacion secuencia que hicimos en el otro notebook  para calcular als diferencias no sé si conveine mas hacerlo como lo tengo ahorita o si es mejor que quede como antes (que se haga la proyeccion segudo al ultimo dato del precio como lo teniamos antes) 
total_velas = len(df_mercado)
corte_idx = int(total_velas * PORCENTAJE_PASADO)
horizonte_futuro = total_velas - corte_idx

print(f"Total Velas: {total_velas}")
print(f"El modelo aprenderá de la vela 0 hasta la {corte_idx}")
print(f"Proyectará una inercia ciega de {horizonte_futuro} velas hacia adelante.")

# Extraemos estrictamente el pasado
df_pasado = df_mercado.iloc[:corte_idx].copy()

### 3. Entrenamiento Ciego (Out-of-Sample)
Aplicamos el Auto-Tuner y descubrimos la Física usando **únicamente** los datos del pasado.

In [ ]:
# =======================================================================
# Función reutilizable: Ejecutar pipeline KineTopus con un drift dado
# =======================================================================
def run_sindy_projection(df_pasado, best_drift, horizonte_futuro):
    """Ejecuta el pipeline completo de KineTopus y retorna la proyección determinista."""
    log_returns, volumen_z, precio_raw, dt_val = MarketLoader.prepare_quant_input(
        df_pasado, disable_returns=False
    )
    t_pasado = np.arange(len(log_returns), dtype=np.float64) * dt_val

    sensor = SpectralAnalyzer()
    fft_results = sensor.analyze(np.column_stack((log_returns, volumen_z)), dt=dt_val)
    b = ContinuousBlender(tolerance=0.0050)
    b.fit(t_pasado, log_returns, fft_results[0]['periods'], 0)
    b.fit(t_pasado, volumen_z, fft_results[1]['periods'], 1)

    r_smooth, r_dot, _ = b.compute_continuous(0, t_pasado)
    v_smooth, v_dot, _ = b.compute_continuous(1, t_pasado)

    detector = RegimeShiftDetector(threshold=5.0, drift=best_drift)
    shift_idx = detector.detect(log_returns, r_smooth)['shift_indices']

    start_regime = shift_idx[-1] if len(shift_idx) > 0 else 0
    end_regime = len(t_pasado)
    if end_regime - start_regime < 15:
        start_regime = shift_idx[-2] if len(shift_idx) > 1 else 0

    disc = PhysicsDiscoverer(poly_degree=1)
    x_matrix = np.column_stack((r_smooth, v_smooth))
    x_dot_matrix = np.column_stack((r_dot, v_dot))

    physics_rep = disc.extract_equations(
        t=t_pasado[start_regime:end_regime], x=x_matrix[start_regime:end_regime],
        x_dot=x_dot_matrix[start_regime:end_regime], dt=dt_val,
        horizon_steps=horizonte_futuro, sigma_res_r=0, sigma_res_v=0,
        last_price=precio_raw[-1], disable_returns=False, disable_norm=False
    )

    pred = physics_rep.get('prediction', {}).get('det_price_path', [])
    r2 = physics_rep.get('score', 0)
    return pred, r2, best_drift

# =======================================================================
# A) Proyección con CUSUMAutoTuner (Clásico - Optimiza R² del pasado)
# =======================================================================
print("═" * 60)
print("🔵 [1/2] Ejecutando CUSUMAutoTuner (Clásico)...")
print("═" * 60)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    tuner_cusum = CUSUMAutoTuner(df_pasado)
    drift_cusum, rep_cusum = tuner_cusum.run_search()
    pred_cusum, r2_cusum, _ = run_sindy_projection(df_pasado, drift_cusum, horizonte_futuro)
print(f"► CUSUM Drift Óptimo: {drift_cusum} | R²: {r2_cusum:.4f}")
print(f"► Proyección generada: {len(pred_cusum)} velas\n")

# =======================================================================
# B) Proyección con PredictiveAutoTuner (Walk-Forward Interno)
# =======================================================================
print("═" * 60)
print("🟠 [2/2] Ejecutando PredictiveAutoTuner (Walk-Forward)...")
print("═" * 60)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    tuner_pred = PredictiveAutoTuner(df_pasado, test_blocks=3)
    drift_pred, rep_pred = tuner_pred.run_search()
    pred_predictive, r2_pred, _ = run_sindy_projection(df_pasado, drift_pred, horizonte_futuro)
print(f"► Predictive Drift Óptimo: {drift_pred} | R²: {r2_pred:.4f}")
print(f"► Proyección generada: {len(pred_predictive)} velas\n")

# =======================================================================
# Resumen comparativo
# =======================================================================
print("═" * 60)
print("📊 RESUMEN COMPARATIVO")
print("═" * 60)
print(f"  {'Método':<25} {'Drift':>8} {'R²':>10} {'Velas Pred.':>12}")
print(f"  {'-'*55}")
print(f"  {'CUSUMAutoTuner':<25} {drift_cusum:>8.1f} {r2_cusum:>10.4f} {len(pred_cusum):>12}")
print(f"  {'PredictiveAutoTuner':<25} {drift_pred:>8.1f} {r2_pred:>10.4f} {len(pred_predictive):>12}")
print(f"  {'-'*55}")
if drift_cusum == drift_pred:
    print("  ⚡ Ambos tuners convergieron al MISMO drift.")
else:
    print(f"  ⚡ Los tuners eligieron drifts DIFERENTES ({drift_cusum} vs {drift_pred}).")

### 4. Inspección Empírica Visual
Superponemos la proyección matemática sobre la trayectoria real del activo que ocurrió después del Corte.

In [ ]:
# Eje X de Tiempo para gráficos
t_total = np.arange(total_velas)
df_mercado['SMA_20'] = df_mercado['Close'].rolling(window=150).mean()

fig = go.Figure()

# 1. El Pasado Conocido (Con lo que se entrenó SINDy)
fig.add_trace(go.Scatter(
    x=t_total[:corte_idx], 
    y=df_mercado['Close'].values[:corte_idx],
    mode='lines',
    line=dict(color='gray', width=2),
    name='Pasado (Contexto)'
))

# 2. El Futuro Real (Lo que realmente ocurrió y que el modelo NO vio)
fig.add_trace(go.Scatter(
    x=t_total[corte_idx:], 
    y=df_mercado['Close'].values[corte_idx:],
    mode='lines',
    line=dict(color='rgba(255, 255, 255, 0.5)', width=2, dash='dash'),
    name='Futuro Real (La Verdad)'
))

# 2.5 Media Movil
fig.add_trace(go.Scatter(
    x=t_total, 
    y=df_mercado['SMA_20'].values,
    mode='lines',
    line=dict(color='orange', width=2, dash='dot'),
    name='Media Móvil (150)'
))

# =============================================
# 3A. Proyección CUSUM (Clásica) — Cyan
# =============================================
if len(pred_cusum) > 0:
    t_fut_cusum = np.arange(corte_idx, corte_idx + len(pred_cusum))
    fig.add_trace(go.Scatter(
        x=t_fut_cusum, 
        y=pred_cusum,
        mode='lines',
        line=dict(color='cyan', width=4),
        name=f'CUSUM (drift={drift_cusum}, R²={r2_cusum:.3f})'
    ))
else:
    print("⚠️ CUSUMAutoTuner: La matemática colapsó, sin proyección.")

# =============================================
# 3B. Proyección Predictive (Walk-Forward) — Magenta
# =============================================
if len(pred_predictive) > 0:
    t_fut_pred = np.arange(corte_idx, corte_idx + len(pred_predictive))
    fig.add_trace(go.Scatter(
        x=t_fut_pred, 
        y=pred_predictive,
        mode='lines',
        line=dict(color='#FF6EC7', width=4, dash='dashdot'),
        name=f'Predictive (drift={drift_pred}, R²={r2_pred:.3f})'
    ))
else:
    print("⚠️ PredictiveAutoTuner: La matemática colapsó, sin proyección.")

# Muro Topológico de la Máquina del Tiempo
fig.add_vline(x=corte_idx, line_dash="solid", line_color="yellow", 
              annotation_text="Día Cero (Corte Simulación)", annotation_position="top left")

fig.update_layout(
    title="Simulador: CUSUMAutoTuner vs PredictiveAutoTuner — Duelo de Proyecciones",
    yaxis_title="Precio Absoluto",
    template='plotly_dark',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Aplicar el parche de auto-escala
hist_min = np.min(df_mercado['Close'].values[:corte_idx])
hist_max = np.max(df_mercado['Close'].values[:corte_idx])
hist_range = hist_max - hist_min
y_min = hist_min - hist_range * 0.5
y_max = hist_max + hist_range * 1.5

fig.update_yaxes(range=[y_min, y_max])

fig.show()

# =============================================
# Métricas de Error vs Futuro Real
# =============================================
actual_future = df_mercado['Close'].values[corte_idx:]

print("\n" + "═" * 60)
print("📐 MÉTRICAS DE ERROR vs FUTURO REAL")
print("═" * 60)

# Naive Forecast (Línea plana)
last_price = df_mercado['Close'].values[corte_idx - 1]
naive_mape = np.mean(np.abs((actual_future - last_price) / actual_future)) * 100
print(f"  {'Naive (Línea Plana)':<25} MAPE: {naive_mape:.2f}%")

for name, pred in [('CUSUMAutoTuner', pred_cusum), ('PredictiveAutoTuner', pred_predictive)]:
    if len(pred) > 0:
        min_len = min(len(pred), len(actual_future))
        p = np.array(pred[:min_len])
        a = actual_future[:min_len]
        mape = np.mean(np.abs((a - p) / a)) * 100
        # Hit ratio direccional
        dir_pred = np.sign(p[-1] - last_price)
        dir_real = np.sign(a[-1] - last_price)
        hit = '✅ Acierto' if dir_pred == dir_real else '❌ Fallo'
        print(f"  {name:<25} MAPE: {mape:.2f}%  | Dirección: {hit}")
    else:
        print(f"  {name:<25} Sin proyección (colapsó)")

print("═" * 60)